In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# FEATURE ENGINEERING

In [ ]:
#Load clean data and split (train/test)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import joblib

# Load cleaned data
df_clean = pd.read_csv('/kaggle/working/clean_data.csv')
df_clean['clean_text'] = df_clean['clean_text'].astype(str)

# Features and labels
X = df_clean['clean_text']
y = df_clean['label']

# Train / test split (stratified, no leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"Train label distribution:\n{y_train.value_counts().sort_index()}")
print(f"Test label distribution:\n{y_test.value_counts().sort_index()}")

# Save splits for later
joblib.dump((X_train, X_test, y_train, y_test), '/kaggle/working/data_splits.pkl')
print("Saved: data_splits.pkl")

# MODEL 1 :- Word n‑grams TF‑IDF

In [ ]:
#Word n‑grams TF‑IDF

from sklearn.feature_extraction.text import TfidfVectorizer

# Word unigrams + bigrams
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=3000,
    stop_words='english',
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_test_word = word_vectorizer.transform(X_test)

print(f"Word features shape (train): {X_train_word.shape}")
print(f"Word features shape (test):  {X_test_word.shape}")

# Save the fitted vectorizer
joblib.dump(word_vectorizer, '/kaggle/working/word_vectorizer.pkl')
print("Saved: word_vectorizer.pkl")

# MODEL 2 :- Character n‑grams TF‑IDF

In [ ]:
#Character n‑grams TF‑IDF

# Character 3-5 grams (character-level, word boundaries considered)
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=2000,
    sublinear_tf=True
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)

print(f"Char features shape (train): {X_train_char.shape}")
print(f"Char features shape (test):  {X_test_char.shape}")

joblib.dump(char_vectorizer, '/kaggle/working/char_vectorizer.pkl')
print("Saved: char_vectorizer.pkl")

# MODEL 3 :- Meta‑features extraction and scaling

In [ ]:
#Meta‑features extraction and scaling

from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix

def extract_meta_features(texts):
    """Extract numerical features from text list."""
    features = []
    for t in texts:
        t = str(t)
        words = t.split()
        feat = {
            'length': len(t),
            'word_count': len(words),
            'caps_ratio': sum(1 for c in t if c.isupper()) / (len(t) + 1),
            'exclamation_count': t.count('!'),
            'question_count': t.count('?'),
            'punct_count': sum(1 for c in t if c in '.,;:'),
            'avg_word_len': np.mean([len(w) for w in words]) if words else 0
        }
        features.append(feat)
    return pd.DataFrame(features)

# Extract
meta_train_df = extract_meta_features(X_train)
meta_test_df = extract_meta_features(X_test)

# Scale
scaler = StandardScaler()
meta_train_scaled = scaler.fit_transform(meta_train_df)
meta_test_scaled = scaler.transform(meta_test_df)

# Convert to sparse matrix (for concatenation)
X_train_meta = csr_matrix(meta_train_scaled)
X_test_meta = csr_matrix(meta_test_scaled)

print(f"Meta features shape (train): {X_train_meta.shape}")
print(f"Meta features shape (test):  {X_test_meta.shape}")

joblib.dump(scaler, '/kaggle/working/meta_scaler.pkl')
print("Saved: meta_scaler.pkl")

# MODEL 4 :- feature fusion

In [ ]:
#Combine all features (final feature matrix)

from scipy.sparse import hstack

# Stack horizontally: word + char + meta
X_train_final = hstack([X_train_word, X_train_char, X_train_meta])
X_test_final = hstack([X_test_word, X_test_char, X_test_meta])

print(f"Final combined feature matrix:")
print(f"  Train shape: {X_train_final.shape}")
print(f"  Test shape:  {X_test_final.shape}")

# Save final feature matrices (optional, for later use)
joblib.dump((X_train_final, X_test_final), '/kaggle/working/feature_matrices.pkl')
print("Saved: feature_matrices.pkl")